# darts_forecast — train the forecaster and read what it predicts

The strategy's **forecaster** is a Darts model. Every `BAR_MINUTES` minutes of the regular session it predicts each
stock's return over the next `HORIZON` bar(s), from:
- the 1-minute detail inside each bar;
- the other names' moves;
- multi-day context;
- the options market;
- the clock.

This notebook trains it the way it would have run live, then asks whether its forecasts are worth anything.
**It reads the TUNE months only** (§1 says exactly which sessions train, validate and are forecast by each fit).

| § | what |
|---|---|
| 0 | the knobs: model, resolution, horizon, training window, retraining cadence, inputs |
| 1 | **which data trains each fit, which validates it, which it forecasts**: the split, the walk-forward, the fit log |
| 2 | **forecast vs outcome**: IC, hit rate, AUC, calibration; by month, name and hour; over time; the strongest forecasts; one sample day |
| 3 | **the retraining study**: train once, every 20 sessions, weekly, daily (plus your `RETRAIN_MINUTES`). Does retraining more often help? Does a model go stale? |
| 4 | **which inputs matter**: permutation importance by input group, out of sample |
| 5 | **the coin-flip null**: the same numbers on markets with no direction to find |
| 6 | other models on the same split (optional) |
| 7 | checks, the verdict and the scorecard |

**Run time.** Every walk-forward is saved on disk in `strategies/darts_forecast/cache/` (`research.cached_forecast`),
and the other darts_forecast notebooks reuse the same files. A file's key holds every forecast knob, the panel's
full content and the version of `strategy.py`, so a saved forecast is only ever reused for exactly these knobs on
exactly this data; a changed knob, a changed bar or a code change runs the walk-forward again. With everything saved
a run takes about a minute. On a first run with the defaults (gbm, 5-minute bars, daily retraining), expect
about:
- 8 minutes for §1 (up to 15 when other jobs share the CPU);
- 5 minutes for §3 (three more schedules; the daily one is §1's);
- half a minute for §4 (one static fit, then re-forecasts with shuffled inputs; never saved);
- 25 minutes for §5 (three more walk-forwards; up to 40 under load).

A TiDE walk-forward takes about as long as gbm, a TFT one about fifteen times as long (§6 has the times and how
to cut them).

**What the earlier study found** (`strategies/burst_scan/DIRECTION_REPORT.md`): 1-minute OHLCV of these names
carried no tradable direction, and the free options features showed no link to next-day direction. This notebook
tests that again with a learned forecaster, and it is the place to test new inputs the same way.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --all-extras  (Darts + torch: the "darts" extra)
import importlib.util
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds what the darts_forecast notebooks share: loading, the option inputs, the disk
# cache of forecasts, the metrics, the studies and the small tables; charts.py the figure helpers.
# signals() never imports them.
_spec = importlib.util.spec_from_file_location(
    "darts_research", strategies.STRATEGIES_DIR / "darts_forecast" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
_spec = importlib.util.spec_from_file_location(
    "darts_charts", strategies.STRATEGIES_DIR / "darts_forecast" / "charts.py"
)
charts = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(charts)
strat, SM = rs.STRAT, rs.SM  # the loaded strategy and strategy.py's own functions

# --- 0 · the knobs ----------------------------------------------------------------------------
P0 = strat.params  # the shipped config.yaml values: every knob below starts from them
MODEL = P0["model"]  # gbm | gbm_clf | linear | tide | tsmixer | tft  (e.g. MODEL = "tide")
BAR_MINUTES = P0["bar_minutes"]  # the forecast resolution: 1, 5, 10, 15 or 30 minutes
HORIZON = P0["horizon"]  # forecast the return over the next HORIZON bars
TRAIN_DAYS = P0["train_days"]  # each fit's window: the last TRAIN_DAYS sessions before it
VAL_DAYS = P0["val_days"]  # tide/tft: the last VAL_DAYS of them = VALIDATION (early stopping)
RETRAIN_MINUTES = P0["retrain_minutes"]  # 390 = every session, 1950 = weekly, 60 = hourly, 0 = once
FINETUNE_EPOCHS = P0["finetune_epochs"]  # tide/tft: later fits fine-tune N epochs (0 = refit)
N_EPOCHS = P0["n_epochs"]  # tide/tft: epochs of a full fit
RECENCY_HALF_LIFE = P0["recency_half_life"]  # sessions after which a sample counts half; 0 = equal
OPTIONS = P0["options"]  # the option inputs: "none" | "iv" | "all"
DEVICE = P0["device"]  # tide/tft: "cpu" | "mps" (Apple GPU)
SEED = P0["seed"]  # model seed: rerun with another to see the seed-to-seed spread (§6)
CADENCES = (0, 7800, 1950, 390)  # §3 in trading minutes: once, every 20 sessions, weekly, daily
# (0, the control, and RETRAIN_MINUTES are always in §3's study)
RUN_IMPORTANCE = True  # §4: input-group importance (one static fit + shuffled re-forecasts)
IMPORTANCE_SEEDS = (0, 1, 2)  # §4: one whole-day shuffle per seed; their spread = Fig 4's error bar
RUN_NULL = True  # §5: the whole walk-forward again on each coin-flip market (cached)
NULL_SEEDS = (0, 1, 2)
COMPARE_MODELS = []  # §6, e.g. ["gbm_clf", "tide"]; gbm_clf / tide ~8 min, tft ~2 h (see §6)
SAMPLE_NAME, SAMPLE_DAY = "MU", None  # §2's sample day (None = the last scored session)

P = {
    **P0,
    "model": MODEL,
    "bar_minutes": BAR_MINUTES,
    "horizon": HORIZON,
    "train_days": TRAIN_DAYS,
    "val_days": VAL_DAYS,
    "retrain_minutes": RETRAIN_MINUTES,
    "finetune_epochs": FINETUNE_EPOCHS,
    "n_epochs": N_EPOCHS,
    "recency_half_life": RECENCY_HALF_LIFE,
    "options": OPTIONS,
    "device": DEVICE,
    "seed": SEED,
}
assert SM["forecast_valid"](P), "these forecast knobs are invalid (strategy.forecast_valid)"
DEEP = MODEL in rs.SM["DEEP"]
changed = {k: v for k, v in P.items() if v != P0[k]}
print(f"forecast knobs: {rs.model_params(P)}")
print(f"changed from config.yaml: {changed or 'nothing'}")

## 1 · Which data trains each fit, which validates it, and which it forecasts (Table 1, Fig 1)

**The split** (`research.WINDOWS`, the same in every darts_forecast notebook):

| window | role | where it is read |
|---|---|---|
| **TUNE** | you may choose every knob above by looking at it | this notebook, `darts_forecast_data`, `darts_forecast` |
| **TEST** | the frozen config, scored **once** | `darts_forecast_diagnostics` (and `darts_forecast` §11, a replay of the frozen trading rules) |
| **HOLDOUT** | read once at the very end | behind `READ_HOLDOUT = True` |

**Inside TUNE the model is trained and tested by walking forward**, the way it would have run live:

1. **Warm-up.** The first 21 sessions only build the daily inputs and the volatility scale. There is no model yet.
2. **Fit.** At each **retrain point** (every `RETRAIN_MINUTES` of trading time; 390 = before every session) a fresh
   model gets the last `TRAIN_DAYS` sessions before that point:
   - **TRAIN**: the sessions it is fitted on (`train_from` → `train_last` in the fit log).
   - **VALIDATION**: TiDE and TFT with `VAL_DAYS > 0` hold back the last `VAL_DAYS` of those sessions. They are
     not fitted; they only decide when training stops (early stopping, a fresh state per fit). The price: the
     freshest `VAL_DAYS` sessions are never learned from. gbm has no VALIDATION slice: it runs a fixed `gbm_iter`
     boosting rounds and nothing is early-stopped.
   - With `FINETUNE_EPOCHS > 0` a deep model is fit in full only once. Every later retrain continues the previous
     weights for a few epochs.
3. **Forecast.** That model forecasts every bar from its retrain point until the next one, each forecast from bars up
   to that bar only. These are the **out-of-sample** forecasts: the model never saw those bars or anything after them.
4. Step forward and repeat. §2–§7 score the forecasts of all fits, stitched together.

**Table 1** lists every fit with its three slices, in sessions: TRAIN, VALIDATION ("—" when there is none) and
FORECAST. With the defaults (gbm, daily retraining) fit *k* trains on the 40 sessions before its session and
forecasts that one session. **Fig 1** draws the same slices: TRAIN (blue), VALIDATION (amber), FORECAST (green).

So no number below is a training-set number. What is *not* out of sample is your choice of knobs: try twenty
settings here and keep the best, and the TUNE result flatters it. That is what the TEST window is for.

- **For gbm the TUNE forecasts play the role of the validation set**, and TEST is the test set.
- **TEST freezes the knobs, not the weights.** On TEST the walk-forward keeps retraining exactly as here, each model
  on the `TRAIN_DAYS` sessions before it. From late March on, its TRAIN slices lie inside TEST. That is how it would
  run live, and each fit still uses only the past.
- **Half-days.** On the NYSE early closes (2025-11-28 and 2025-12-24 inside TUNE) the stored "regular" panel still
  holds bars after 13:00. The grid ignores them (`strategy.HALF_DAYS`): those sessions end at 13:00 for the inputs,
  the targets and the scoring.

In [ ]:
display(rs.windows_table())
t0 = time.time()
panel = rs.load(settings, engine)  # TUNE only, with the option inputs attached
RT_BP = rs.round_trip_bp(settings, panel)  # one round trip at the strategy's $2,000 orders, in bp
SESS = pd.Index(sorted({str(d) for d in panel.close.index.tz_convert(panel.tz).date}))
print(
    f"TUNE panel: {', '.join(panel.symbols)} · {len(SESS)} sessions {SESS[0]} → {SESS[-1]} · "
    f"{len(panel.close):,} one-minute bars · loaded in {time.time() - t0:.0f} s · "
    f"round trip {RT_BP:.1f} bp"
)
opt = {k[4:]: v for k, v in sorted((panel.features or {}).items()) if k.startswith("opt_")}
print(
    "option inputs (share of name-sessions with a value): "
    + (", ".join(f"{k} {np.isfinite(v.to_numpy()).mean():.0%}" for k, v in opt.items()) or "none")
)

t0 = time.time()
fc = rs.cached_forecast(panel, P)  # the walk-forward, or its saved copy (same knobs, same panel)
inp, log, g = fc["inputs"], fc["log"], fc["inputs"]["g"]
n_fc = int(np.isfinite(fc["edge"]).sum())
n_scored = len(rs.table(fc, "tune"))
last = "last bar of a session has" if HORIZON == 1 else f"last {HORIZON} bars of a session have"
print(
    f"walk-forward ready in {time.time() - t0:.0f} s: {len(log)} fits from "
    f"{log['session'].iloc[0]} to {log['session'].iloc[-1]}, "
    f"{log['seconds'].sum() / 60:.1f} min of fitting ({log['seconds'].mean():.1f} s per fit), "
    f"{log['samples'].median():,.0f} training rows per fit, {n_fc:,} out-of-sample forecasts "
    f"({n_scored:,} scored: the {last} no outcome inside it)"
)

# per fit, in session numbers: TRAIN = train_from .. train_last; VALIDATION = the sessions after
# train_last and before the retrain session; FORECAST = the sessions its forecasts fall in
pos = {d: i for i, d in enumerate(SESS)}  # session date -> session number (= the grid's code)
fit_s = log["session"].map(pos)
tr_a, tr_b = log["train_from"].map(pos), log["train_last"].map(pos)
bars_, names_ = np.nonzero(np.isfinite(fc["fit"]))
made = pd.DataFrame(
    {"fit": fc["fit"][bars_, names_].astype(int), "s": g["code"][bars_], "b": bars_}
)
fc_a, fc_b = made.groupby("fit")["s"].min(), made.groupby("fit")["s"].max()

table1 = pd.DataFrame(
    {
        "retrain at": log["session"].to_numpy(),
        "TRAIN (fitted)": [rs.sessions_span(a, b, SESS) for a, b in zip(tr_a, tr_b, strict=True)],
        "VALIDATION (early stopping only)": [
            rs.sessions_span(b + 1, s - 1, SESS) for b, s in zip(tr_b, fit_s, strict=True)
        ],
        "FORECAST (out of sample)": [rs.sessions_span(fc_a[k], fc_b[k], SESS) for k in log["fit"]],
        "training rows": log["samples"].to_numpy(),
        "full fit": log["fresh"].to_numpy(),
        "seconds": log["seconds"].to_numpy(),
    },
    index=pd.Index(log["fit"].to_numpy(), name="fit"),
)
a0, b0, s0 = tr_a.iloc[0], tr_b.iloc[0], fit_s.iloc[0]
parts = [
    f"warm-up {SESS[0]} → {SESS[a0 - 1]} ({a0} sessions: inputs only, no model)" if a0 else "",
    f"first TRAIN {rs.sessions_span(a0, b0, SESS)}",
    f"its VALIDATION {rs.sessions_span(b0 + 1, s0 - 1, SESS)}" if b0 + 1 < s0 else "no VALIDATION",
    f"scored forecasts {SESS[s0]} → {SESS[-1]} ({len(SESS) - s0} sessions, each from a model "
    "fit only on bars before it)",
    "TEST and HOLDOUT: not loaded",
]
print("The split in this run: " + " · ".join(x for x in parts if x))
print("Table 1 · every fit: the sessions it trains on, validates on and forecasts (n sessions)")
display(table1)

groups = {}
for name, grp in inp["groups"].items():
    groups.setdefault(grp, []).append(name)
what = {
    "intraday": "from the 1-minute bars inside each grid bar, known at its close",
    "market": "the other names' average return in the same bar",
    "daily": "previous sessions (gap: today's first open), repeated on every bar of the day",
    "options": "end of day, known before the session (never staler than 5 days)",
    "flag": "1 = that option source has data this session (a missing input enters as 0: "
    "Darts refuses NaN)",
}
table1b = pd.DataFrame(
    [
        {"group": k, "n": len(v), "inputs": ", ".join(v), "known": what.get(k, "")}
        for k, v in groups.items()
    ]
    + [
        {
            "group": "target lags",
            "n": P["lookback"],
            "inputs": "the return being forecast, last bars",
            "known": "at each bar's close",
        },
        {
            "group": "future",
            "n": len(inp["future_names"]),
            "inputs": ", ".join(inp["future_names"]),
            "known": "the calendar, in advance",
        },
        {
            "group": "static",
            "n": len(inp["names"]),
            "inputs": "which name (one-hot)",
            "known": "one global model for all names",
        },
    ]
).set_index("group")
print("Table 1b · what the model sees at every decision bar, per name")
with pd.option_context("display.max_colwidth", 200):
    display(table1b)

In [ ]:
# Fig 1: fit time per retrain, and the TRAIN / VALIDATION / FORECAST slices of every fit (Table 1).
wall = panel.close.index.tz_convert(panel.tz).tz_localize(None)  # New York wall clock
at = lambda b: wall[g["start"][b]]  # noqa: E731  the time grid bar b starts
sf, last_b = g["sess_first"], made.groupby("fit")["b"].max()  # last_b: each fit's last forecast
step = max(1, int(np.ceil(len(log) / 120)))  # at most ~120 rows in the chart
spans = []
for i in range(0, len(log), step):
    T, k = int(log["retrain_bucket"].iloc[i]), int(log["fit"].iloc[i])
    lo = int(sf[tr_a.iloc[i]])  # TRAIN starts at train_from's open ...
    fit_hi = int(sf[tr_b.iloc[i] + 1]) if tr_b.iloc[i] < fit_s.iloc[i] else T  # ... ends here
    done = wall[g["end"][last_b[k]]] + pd.Timedelta(minutes=1)  # the close of its last bar
    spans.append((k + 1, at(lo), at(fit_hi), at(T), done))
ms = lambda a, b: (b - a).total_seconds() * 1000  # noqa: E731  bar length in milliseconds

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.22, 0.78],
    vertical_spacing=0.06,
    subplot_titles=(
        "seconds per fit",
        "each row = one fit: what it trains on, and what it forecasts",
    ),
)
fresh = log["fresh"].astype(bool).to_numpy()
for mask, name, colr in (
    (fresh, "full fit", plots.SERIES[0]),
    (~fresh, "fine-tune", plots.SERIES[4]),
):
    if mask.any():
        fig.add_trace(
            go.Scatter(
                x=[at(int(b)) for b in log["retrain_bucket"][mask]],
                y=log["seconds"][mask],
                mode="markers",
                name=name,
                marker={"color": colr, "size": 5},
                showlegend=bool((~fresh).any()),
            ),
            row=1,
            col=1,
        )
for i, (label, colr) in enumerate(
    (
        ("TRAIN", plots.SERIES[0]),
        ("VALIDATION", plots.SERIES[2]),
        ("FORECAST (out of sample)", plots.SERIES[1]),
    )
):
    rows = [(f, a, b) for f, *x in spans for a, b in [(x[i], x[i + 1])] if b > a]
    if rows:
        fig.add_trace(
            go.Bar(
                y=[f for f, _, _ in rows],
                x=[ms(a, b) for _, a, b in rows],
                base=[a.isoformat() for _, a, _ in rows],
                orientation="h",
                name=label,
                marker={"color": colr, "line": {"width": 0}},
                width=0.9,
                customdata=[[f"{a:%Y-%m-%d %H:%M}", f"{b:%Y-%m-%d %H:%M}"] for _, a, b in rows],
                hovertemplate=label + " · fit %{y}<br>%{customdata[0]} → %{customdata[1]}"
                "<extra></extra>",
            ),
            row=2,
            col=1,
        )
first_lo = spans[0][1]
fig.add_vrect(
    x0=wall[0],
    x1=first_lo,
    fillcolor=plots._MUTED,
    opacity=0.15,
    line_width=0,
    layer="below",
    annotation_text="warm-up",
    annotation_position="top left",
    row=2,
    col=1,
)
a, b = rs.WINDOWS["test"]
fig.add_vrect(
    x0=a,
    x1=b,
    fillcolor=plots.SERIES[5],
    opacity=0.08,
    line_width=0,
    layer="below",
    annotation_text="TEST (not read here)",
    annotation_position="top left",
    row=2,
    col=1,
)
fig.add_vline(
    x=rs.WINDOWS["holdout"][0],
    line={"color": plots.SERIES[5], "dash": "dot", "width": 1},
    row=2,
    col=1,
)
fig.add_annotation(
    x=rs.WINDOWS["holdout"][0],
    y=1,
    xref="x2",
    yref="y2 domain",
    text="HOLDOUT →",
    showarrow=False,
    xanchor="left",
    font={"color": plots._MUTED, "size": 11},
)
fig.update_layout(barmode="overlay", hovermode="closest")
fig.update_yaxes(autorange="reversed", title_text="fit", row=2, col=1)
fig.update_yaxes(title_text="s", rangemode="tozero", row=1, col=1)
fig.update_xaxes(
    type="date", range=[wall[0] - pd.Timedelta(days=3), pd.Timestamp(b) + pd.Timedelta(days=35)]
)
plots._base(
    fig,
    title=f"Fig 1 · the walk-forward: {len(log)} fits"
    + (f" (one in {step} shown)" if step > 1 else "")
    + f", training window = last {TRAIN_DAYS} sessions"
    + (f" (the last {VAL_DAYS} = VALIDATION)" if DEEP and VAL_DAYS > 0 else "")
    + f", retrain every {RETRAIN_MINUTES} min",
    height=int(min(1100, 360 + 6 * len(spans))),
)
fig.update_layout(margin={"t": 90})
charts.below(fig, -0.06)
fig.show()

## 2 · Forecast vs outcome on TUNE (Table 2, Fig 2a–2e)

Each scored row is one forecast for one stock at one decision bar. Next to it is what actually happened:

| column | what |
|---|---|
| `edge`, `edge_bp` | the model's expected return over the next `HORIZON` bar(s), in volatility units and in bp |
| `real`, `real_bp` | the realized return over the same bars |

A row is scored when the forecast exists and its outcome finishes inside the same session (on a half-day,
by the 13:00 close).

**How to read Table 2:**

| metric | what it asks | no skill | worth a look |
|---|---|---|---|
| **ic** | rank correlation of forecast and outcome | 0 | > 0.02 with t ≥ 2, and above §5's null |
| **t** | the IC in standard errors. Clustered by session: the five names share each day's market, so they are not five independent tests | \|t\| ≥ 2 by chance 1 time in 20 | ≥ 2 |
| **hit** | share of forecasts with the right sign | 0.5 | 0.51–0.52 is already a lot at 5-minute bars |
| **auc** | P(the forecast ranks a random up-bar above a random down-bar) | 0.5 | > 0.51 |
| **mse_skill** | 1 − forecast error ÷ the error of always forecasting 0 | ≤ 0 | > 0. A forecast that is mostly noise is *worse* than forecasting 0 |
| **sign_bp** | bp per forecast from going long when it is up and short when it is down, held `HORIZON` bar(s), before costs | 0 | > the round trip |
| **net_bp** | sign_bp minus one round trip of costs (`RT_BP` = 14 bp: commission with its \$1 minimum, spread and slippage at the strategy's \$2,000 orders), as if every forecast were traded | < 0 | > 0 |

`net_bp` is the crude economic test. The strategy trades only bars with a large forecast (`edge_bp`), so Table 2c
repeats it for the strongest forecasts only.

In [ ]:
t = rs.table(fc, "tune")
s2 = rs.summary(t, cost_bp=RT_BP)
MEANING = {
    "ic": "rank correlation of forecast and outcome (0 = none)",
    "t": "the IC in standard errors, clustered by session",
    "n": "scored forecasts",
    "days": "sessions",
    "hit": "share with the right sign (0.5 = coin flip)",
    "auc": "P(an up-bar is ranked above a down-bar) (0.5 = coin flip)",
    "mse_skill": "1 - error / error of forecasting 0 (> 0 = better than 0)",
    "sign_bp": "bp per forecast trading its sign (long if up, short if down), before costs",
    "sign_t": "sign_bp in standard errors, clustered by session",
    "net_bp": f"sign_bp minus a {RT_BP:.1f} bp round trip, every forecast traded",
}
fmt = lambda k, v: f"{v:,.0f}" if k in ("n", "days") else f"{v:.4f}"  # noqa: E731
table2 = pd.DataFrame(
    [{"metric": k, "value": fmt(k, v), "meaning": MEANING.get(k, "")} for k, v in s2.items()]
).set_index("metric")
print(f"Table 2 · {MODEL} on TUNE, {BAR_MINUTES}-minute bars, horizon {HORIZON}")
display(table2)
big = float((t["edge_bp"].abs() > P["edge_bp"]).mean())
print(
    f"{big:.1%} of the forecasts are larger than the trading rule's edge_bp = {P['edge_bp']:g} bp "
    f"(only these bars can be traded; one round trip costs {RT_BP:.1f} bp)"
)

**Fig 2a: calibration.** All forecasts are sorted into ten equal groups, from the most bearish (D1) to the most
bullish (D10).
- **Line:** the mean forecast of each group.
- **Bars:** the mean realized return, ± 2 standard errors clustered by session.

A forecaster that knows something has bars rising from D1 to D10, close to the line. Flat bars mean no edge. Bars
that rise much less than the line mean an over-confident model.

In [ ]:
cal = rs.calibration(t)
dec = pd.qcut(t["edge"].rank(method="first"), 10, labels=False)
err = []
for q in cal.index:
    r = rs.day_t(t.loc[dec == q, "real_bp"], t.loc[dec == q, "date"])
    err.append(2 * abs(r["mean"] / r["t"]) if r["t"] and np.isfinite(r["t"]) else np.nan)
xs = [f"D{q + 1}" for q in cal.index]
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=xs,
        y=cal["real_bp"],
        name="realized return (bp) ± 2 SE",
        marker_color=plots.SERIES[0],
        error_y={"type": "data", "array": err, "color": plots._MUTED},
    )
)
fig.add_trace(
    go.Scatter(
        x=xs,
        y=cal["edge_bp"],
        mode="lines+markers",
        name="mean forecast (bp)",
        line={"color": plots.SERIES[2], "width": 2},
    )
)
fig.add_hline(y=0, line={"color": plots._INK, "width": 1, "dash": "dot"})
plots._base(
    fig,
    title="Fig 2a · calibration: forecast deciles, mean forecast vs mean realized return",
    height=400,
)
fig.update_xaxes(title_text="forecast decile (D1 = most bearish, D10 = most bullish)")
fig.update_yaxes(title_text="bp over the next bar(s)")
charts.below(fig, -0.22)
fig.show()
display(cal.assign(se2_bp=err).set_axis(xs).rename_axis("decile").round(3))

**Fig 2b / Table 2b: where the forecast works, if anywhere.** The same IC and signed return are shown:
- per month;
- per name;
- per hour of the day (New York; the hour the decision bar closes in).

Error bars are ± 2 standard errors. A real effect shows up in most months and most names. One good month or one
good name among many is what luck looks like.

In [ ]:
tabs = {}
fig = make_subplots(
    rows=2,
    cols=3,
    vertical_spacing=0.16,
    horizontal_spacing=0.06,
    subplot_titles=(
        "IC by month",
        "IC by name",
        "IC by hour",
        "signed bp by month",
        "signed bp by name",
        "signed bp by hour",
    ),
)
for j, key in enumerate(("month", "name", "hour"), start=1):
    b = rs.by(t, key, cost_bp=RT_BP).astype(float)
    tabs[key] = b
    lab = [f"{int(h):02d}:00" for h in b.index] if key == "hour" else [str(i) for i in b.index]
    for row, col, tcol, colr, name in (
        (1, "ic", "t", plots.SERIES[0], "rank IC ± 2 SE"),
        (2, "sign_bp", "sign_t", plots.SERIES[1], "signed bp per forecast ± 2 SE (before costs)"),
    ):
        se = (b[col] / b[tcol]).abs().replace(np.inf, np.nan)
        fig.add_trace(
            go.Bar(
                x=lab,
                y=b[col],
                marker_color=colr,
                name=name,
                showlegend=j == 1,
                error_y={"type": "data", "array": 2 * se, "color": plots._MUTED},
            ),
            row=row,
            col=j,
        )
plots._base(fig, title="Fig 2b · the forecast by month, by name and by hour of the day", height=620)
fig.update_layout(margin={"t": 90})
charts.below(fig, -0.1)
fig.show()
print("Table 2b · the same as numbers")
display(
    pd.concat(
        {k: v[["n", "ic", "t", "hit", "auc", "sign_bp", "sign_t"]] for k, v in tabs.items()}
    ).round(3)
)

**Fig 2c: the IC over time.** Gray dots are one session's IC each. The line is their 10-session average
(`research.rolling_ic`). The gray band is ± 2 standard errors of a 10-session average for a model with no skill.

A model that "works while the regime lasts" would show long stretches outside the band. Wandering inside it is noise.

In [ ]:
dic = rs.rolling_ic(t, 1)  # the raw per-session IC
dic.index = pd.to_datetime(dic.index)
roll = rs.rolling_ic(t, 10)
roll.index = pd.to_datetime(roll.index)
band = 2 * float(dic.std()) / np.sqrt(10)
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=dic.index,
        y=dic,
        mode="markers",
        name="IC of one session",
        marker={"color": plots._MUTED, "size": 4},
    )
)
fig.add_trace(
    go.Scatter(
        x=roll.index,
        y=roll,
        mode="lines",
        name="10-session average",
        line={"color": plots.SERIES[0], "width": 2},
    )
)
fig.add_hrect(y0=-band, y1=band, fillcolor=plots._MUTED, opacity=0.12, line_width=0, layer="below")
fig.add_hline(y=0, line={"color": plots._INK, "width": 1, "dash": "dot"})
fig.update_xaxes(type="date")
plots._base(
    fig,
    title="Fig 2c · rank IC over time (band: ± 2 SE of a 10-session average, no skill)",
    height=380,
)
charts.below(fig, -0.15)
fig.show()
print(
    f"{(dic > 0).mean():.0%} of {dic.notna().sum()} sessions have a positive IC (50% = coin flip); "
    f"{(roll.dropna().abs() > band).mean():.0%} of the 10-session averages lie outside the band "
    f"(about 5% expected by chance; overlapping averages make this share jumpy)"
)

**Fig 2d: how big the forecasts are.** Left: the distribution of the forecasts against the distribution of what
actually happened, on a log scale. Right, for models that give one: the forecast probability that the price goes up.

A model with no information learns to forecast close to 0, which is the best guess when nothing is predictable. Its
forecasts then sit in a narrow spike inside a much wider distribution of outcomes.

In [ ]:
lim = float(np.nanpercentile(t["real_bp"].abs(), 99))
bins = np.linspace(-lim, lim, 121)
mid = (bins[1:] + bins[:-1]) / 2
has_p = bool(np.isfinite(t["p_up"]).any())
fig = make_subplots(
    rows=1,
    cols=2 if has_p else 1,
    subplot_titles=("forecast vs realized (bp, density, log scale)", "P(up)") if has_p else None,
)
for col, name, colr, op in (
    ("real_bp", "realized (bp)", plots.SERIES[0], 0.55),
    ("edge_bp", "forecast (bp)", plots.SERIES[2], 0.85),
):
    dens = np.histogram(t[col], bins, density=True)[0]  # values beyond ±lim are left out
    fig.add_trace(go.Bar(x=mid, y=dens, name=name, marker_color=colr, opacity=op), row=1, col=1)
fig.update_yaxes(type="log", row=1, col=1)
if has_p:
    pb = np.linspace(0, 1, 51)
    dens = np.histogram(t["p_up"].dropna(), pb, density=True)[0]
    fig.add_trace(
        go.Bar(x=(pb[1:] + pb[:-1]) / 2, y=dens, name="P(up)", marker_color=plots.SERIES[4]),
        row=1,
        col=2,
    )
fig.update_layout(barmode="overlay", bargap=0)
plots._base(
    fig,
    title="Fig 2d · the size of the forecasts against the size of the outcomes "
    "(bp, the central 99% of outcomes; density, log scale)",
    height=380,
)
if has_p:
    fig.update_layout(margin={"t": 90})  # room for the subplot titles under the title
charts.below(fig, -0.15)
fig.show()
ratio = float(t["edge_bp"].std() / t["real_bp"].std())
print(
    f"forecast spread (std) {t['edge_bp'].std():.2f} bp vs realized {t['real_bp'].std():.2f} bp: "
    f"the forecasts vary {ratio:.0%} as much as the outcomes"
    + ("" if has_p else f" ({MODEL} gives no P(up))")
)

**Table 2c: the strongest forecasts only.** The trading rule ignores small forecasts, so what matters is the tails.
For the strongest x% of forecasts (by |forecast|), the table shows the signed return per bar, its t and the result
after a round trip.

This is descriptive: the cut-off is read from all of TUNE at once. The strategy notebook's `edge_bp` rule is the
causal version.

In [ ]:
rows = []
for q in (1.0, 0.5, 0.2, 0.1, 0.05, 0.01):
    cut = float(t["edge_bp"].abs().quantile(1 - q)) if q < 1 else 0.0
    sel = t[t["edge_bp"].abs() >= cut]
    r = rs.day_t(np.sign(sel["edge"]) * sel["real_bp"], sel["date"])
    rows.append(
        {
            "strongest": f"{q:.0%}",
            "|forecast| from bp": cut,
            "forecasts": len(sel),
            "per session": len(sel) / t["date"].nunique(),
            "hit": float((np.sign(sel["edge"]) == np.sign(sel["real"]))[sel["real"] != 0].mean()),
            "sign_bp": r["mean"],
            "t": r["t"],
            "net_bp": r["mean"] - RT_BP,
        }
    )
table2c = pd.DataFrame(rows).set_index("strongest")
print(f"Table 2c · trading only the strongest forecasts (round trip {RT_BP:.1f} bp)")
display(table2c.round(3))

**Fig 2e: one sample day** (`SAMPLE_NAME`, `SAMPLE_DAY`). Top: the 1-minute price. Bottom: at each decision bar
(drawn at the time it closes), the forecast for the next `HORIZON` bar(s) and what actually happened over them.
This is what the model "says" during a session.

In [ ]:
day = SAMPLE_DAY or t["date"].max()
name = SAMPLE_NAME if SAMPLE_NAME in inp["names"] else inp["names"][0]
d = t[(t["name"] == name) & (t["date"] == day)].sort_values("minute")
px = panel.close[name].tz_convert(panel.tz)
px = px[px.index.strftime("%Y-%m-%d") == day]
px.index = px.index.tz_localize(None) + pd.Timedelta(minutes=1)  # stamped at each minute's close
when = pd.DatetimeIndex(pd.Timestamp(day) + pd.to_timedelta(d["minute"].to_numpy(), unit="min"))
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.45, 0.55],
    vertical_spacing=0.1,
    subplot_titles=(
        f"{name} on {day}: one-minute close",
        f"each decision bar: forecast for the next {HORIZON} bar(s) vs what happened (bp)",
    ),
)
fig.add_trace(
    go.Scatter(
        x=px.index, y=px, mode="lines", name="close", line={"color": plots._INK, "width": 1.5}
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=when,
        y=d["real_bp"].to_numpy(),
        name="what happened next (bp)",
        marker_color=plots.SERIES[0],
        opacity=0.6,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=when,
        y=d["edge_bp"].to_numpy(),
        mode="lines+markers",
        name="forecast (bp)",
        line={"color": plots.SERIES[2], "width": 2},
        marker={"size": 4},
    ),
    row=2,
    col=1,
)
fig.update_xaxes(type="date")
plots._base(fig, title=f"Fig 2e · {name} on {day}: what the model said, bar by bar", height=540)
fig.update_layout(margin={"t": 90})
charts.below(fig, -0.1)
fig.show()
r = rs.rank_ic(d) if len(d) >= 30 else {"ic": d["edge"].corr(d["real"], method="spearman")}
print(f"{len(d)} forecasts for {name} on {day}; that day's IC for this name: {r['ic']:.3f}")

## 3 · The retraining study (Table 3, Table 3b, Fig 3a, Fig 3b)

The same model, the same inputs and the same forecast bars, retrained on different schedules (`CADENCES`, in
minutes of trading time; 390 = one session), by `research.retrain_study`:

| cadence | meaning |
|---|---|
| 0 | trained **once** at the first retrain point and never again: the control |
| 7800 | every 20 sessions (about monthly) |
| 1950 | every 5 sessions (weekly) |
| 390 | before every session (the default) |
| 60, 30 … | intraday: add them to `CADENCES` (each costs one fit per step: 60 = 6–7 fits a day) |

The control (0) and `RETRAIN_MINUTES` from §0 are always in the study.

Every schedule starts at the same first retrain point and forecasts the same bars (a §7 check). The comparison is
therefore **paired**: same bars, same outcomes, only the model's freshness differs.

**How to read it:**
- **Does retraining help at all?** Compare each row's IC with the once-trained model's. Table 3b gives the
  per-session difference and its t: \|t\| < 2 means no detectable difference. It does so twice:
  - the **per-session IC** ranks all of a session's forecasts at once, so it scores picking the right names AND
    timing the day's moves;
  - the **cross-sectional IC** ranks the five names against each other at each bar, which removes the market's
    move and the model's overall forecast level: picking names only.

  With intraday retraining one session is forecast by several models whose forecast level jumps at each refit.
  That moves the per-session IC but not the cross-sectional one, so read both.
- **Does a model go stale?** Fig 3b plots the IC against the **age of the model** that made the forecast, in
  sessions since its fit. If "the current regime lasts a few days" held here, an old model's IC would fall with age.
  The daily model's forecasts are all age 0. Within one schedule, age is also the date: the once-trained model's
  "40+" bucket is simply the last months of TUNE. So a fall with age can also be a change of market. Table 3b,
  which compares schedules on the same sessions, does not have this problem. Read the `sessions` column too: the
  once-trained model's first age buckets hold one to five sessions, too few for a reliable error bar.
- **Beware of noise.** The best of several noisy ICs beats the others by luck alone, and Table 3b runs two tests per
  schedule. A difference only counts if Table 3b's t is clearly larger than 2, and if it holds on TEST.
- **Cost.** More frequent retraining costs fit time (the `fit_minutes` column: the schedule's total). For TiDE /
  TFT, fine-tuning (`FINETUNE_EPOCHS`) is what makes daily retraining affordable.

In [ ]:
CADS = tuple(dict.fromkeys([0, *CADENCES, RETRAIN_MINUTES]))  # the control and §0's own cadence
t0 = time.time()
rt, rt_tables = rs.retrain_study(panel, P, CADS, "tune", cost_bp=RT_BP)  # cached per cadence
rt.insert(0, "schedule", [rs.cadence_label(c) for c in rt.index])
rt["fit_minutes"] = rt["fit_seconds"] / 60
print(f"Table 3 · the same {MODEL} retrained on {len(CADS)} schedules ({time.time() - t0:.0f} s)")
display(
    rt[
        ["schedule", "fits", "fit_minutes", "n", "ic", "t", "hit", "auc", "sign_bp", "net_bp"]
    ].round(4)
)

ref = CADS[0]  # 0: trained once, the control
dics = {c: rs.rolling_ic(tb, 1) for c, tb in rt_tables.items()}  # the per-session IC
xics = {c: rs.xs_ic(tb) for c, tb in rt_tables.items()}
rows = []
for c in CADS[1:]:
    diff, xdiff = (dics[c] - dics[ref]).dropna(), (xics[c] - xics[ref]).dropna()
    r = rs.day_t(diff.to_numpy(), diff.index.to_numpy())
    rx = rs.day_t(xdiff.to_numpy(), xdiff.index.to_numpy())
    rows.append(
        {
            "schedule": rs.cadence_label(c),
            "vs": rs.cadence_label(ref),
            "per-session IC difference": r["mean"],
            "t": r["t"],
            "cross-sectional IC difference": rx["mean"],
            "t (cross-sectional)": rx["t"],
            "sessions": r["n"],
            "sessions better (per-session IC)": float((diff > 0).mean()),
        }
    )
table3b = pd.DataFrame(rows).set_index("schedule")
print(f"Table 3b · each schedule against '{rs.cadence_label(ref)}', paired by session (same bars)")
display(table3b.round(4))
tmax = table3b[["t", "t (cross-sectional)"]].abs().max(axis=1)
RETRAIN_DIFFERS = bool((tmax >= 2).any())
RETRAIN_WORDS = (
    f"{', '.join(tmax.index[tmax >= 2])} differ(s) from training once at |t| ≥ 2"
    if RETRAIN_DIFFERS
    else f"no schedule differs from training once beyond noise (largest paired |t| "
    f"{tmax.max():.2f})"
)
print(
    f"Retraining cadence: {RETRAIN_WORDS}. With {2 * len(tmax)} paired tests, one |t| ≥ 2 turns "
    f"up by luck alone up to {1 - 0.95 ** (2 * len(tmax)):.0%} of the time: it counts only if it "
    "holds on TEST."
)

fig = go.Figure()
for i, c in enumerate(CADS):
    r = rs.rolling_ic(rt_tables[c], 10)
    r.index = pd.to_datetime(r.index)
    fig.add_trace(
        go.Scatter(
            x=r.index,
            y=r,
            mode="lines",
            name=rs.cadence_label(c),
            line={"color": plots.SERIES[i % len(plots.SERIES)], "width": 2},
        )
    )
fig.add_hrect(y0=-band, y1=band, fillcolor=plots._MUTED, opacity=0.12, line_width=0, layer="below")
fig.add_hline(y=0, line={"color": plots._INK, "width": 1, "dash": "dot"})
fig.update_xaxes(type="date")
plots._base(
    fig, title="Fig 3a · 10-session IC per retraining schedule (band: no-skill noise)", height=400
)
charts.below(fig, -0.15)
fig.show()

In [ ]:
# Fig 3b: IC by the age of the model that made the forecast (sessions since its fit).
edges, names = [0, 1, 5, 10, 20, 40, 10_000], ["0", "1-4", "5-9", "10-19", "20-39", "40+"]
fig, rows = go.Figure(), []
for i, c in enumerate(CADS):
    tb = rt_tables[c]
    # a fit is "born" at the session of its first forecast; age = sessions since then
    born = tb.groupby("fit")["date"].transform("min").map(pos)
    age = pd.cut(tb["date"].map(pos) - born, edges, right=False, labels=names)
    for a_, part in tb.groupby(age, observed=True):
        r = rs.rank_ic(part)
        rows.append(
            {
                "schedule": rs.cadence_label(c),
                "age": a_,
                "ic": r["ic"],
                "t": r["t"],
                "n": r["n"],
                "sessions": r["days"],
            }
        )
    sub = pd.DataFrame([x for x in rows if x["schedule"] == rs.cadence_label(c)])
    fig.add_trace(
        go.Scatter(
            x=sub["age"].astype(str),
            y=sub["ic"],
            mode="lines+markers",
            name=rs.cadence_label(c),
            line={"color": plots.SERIES[i % len(plots.SERIES)], "width": 2},
            error_y={
                "type": "data",
                "array": 2 * (sub["ic"] / sub["t"]).abs(),
                "color": plots.SERIES[i % len(plots.SERIES)],
                "thickness": 1,
            },
        )
    )
fig.add_hline(y=0, line={"color": plots._INK, "width": 1, "dash": "dot"})
fig.update_xaxes(
    categoryorder="array", categoryarray=names, title_text="model age (sessions since its fit)"
)
plots._base(
    fig, title="Fig 3b · does a model go stale? IC by the age of the model (± 2 SE)", height=400
)
charts.below(fig, -0.22)
fig.show()
display(pd.DataFrame(rows).set_index(["schedule", "age"]).round(4))

## 4 · Which inputs matter (Table 4, Fig 4)

This is permutation importance by **input group**, strictly out of sample (`research.group_importance`):

1. **One static model** is fit at the first retrain point, exactly like §3's "once" schedule: on the TRAIN slice of
   the first fit (Table 1, row 0). Every later TUNE bar is out of sample for it.
2. It forecasts all of those bars, in two ways:
   - **as is**: the base IC, the same forecasts as §3's "once" row (a §7 check);
   - **once per input group, with that group's values moved to other days**: one random permutation of the sessions
     after the fit, applied whole days at a time, so each day's intraday shape stays but its link to that day's
     outcomes is broken.
3. **IC drop** = the base IC minus the IC with the group shuffled. One shuffle per seed in `IMPORTANCE_SEEDS`: Fig 4 shows
   their mean, with their standard deviation as the error bar.
   - **positive:** the model uses the group, and it helps out of sample;
   - **about 0:** the group does not matter;
   - **negative:** the group hurts out of sample (the model learned noise from it).

The groups are the rows of Table 1b plus **target lags**: the model's own lagged values of the return it forecasts.
A static model can only use inputs that moved during its TRAIN slice. The open-interest inputs (MarketData, from
2025-10-06) are missing throughout the default one, so they and their `has_oi` flag cannot matter to it; the printout
lists every such input.

**If the base IC is about 0, every drop is about 0 ± noise:** there is no skill to lose. Importance only means
something for a model that has an edge.

In [ ]:
if RUN_IMPORTANCE:
    t0 = time.time()
    runs = {s: rs.group_importance(panel, P, repeats=1, seed=s) for s in IMPORTANCE_SEEDS}
    imp_base = runs[IMPORTANCE_SEEDS[0]].attrs["base_ic"]  # the static model, nothing shuffled
    drops = pd.DataFrame({s: r.set_index("group")["ic_drop"] for s, r in runs.items()})
    imp = (
        runs[IMPORTANCE_SEEDS[0]]
        .set_index("group")[["inputs"]]
        .assign(ic_drop=drops.mean(axis=1), spread=drops.std(axis=1))
        .sort_values("ic_drop")
    )
    once = rt.loc[0]  # §3's "once" row: the same static model's forecasts
    print(
        f"Table 4 · one static {MODEL} fit at {log['session'].iloc[0]} (TRAIN "
        f"{log['train_from'].iloc[0]} → {log['train_last'].iloc[0]}), scored on every later TUNE "
        f"bar: base IC {imp_base:.4f} · {len(IMPORTANCE_SEEDS)} shuffle(s) per group · "
        f"{time.time() - t0:.0f} s"
    )
    display(imp.round(4))
    lo, hi = int(g["sess_first"][tr_a.iloc[0]]), int(g["sess_first"][tr_b.iloc[0] + 1])
    flat = [  # inputs that never move inside the static model's TRAIN slice: it cannot use them
        nm
        for i, nm in enumerate(inp["past_names"])
        if all(np.ptp(np.nan_to_num(inp["past"][s][lo:hi, i])) == 0 for s in inp["names"])
    ]
    print(f"constant over the static model's TRAIN slice (it cannot use them): {', '.join(flat)}")
    print(
        f"Read with care: the base IC is §3's 'once' IC, {once['ic']:.4f} with t {once['t']:.1f}"
        + (" — no detectable skill to lose, so every drop is noise." if abs(once["t"]) < 2 else ".")
        + " The drops only mean something for a model whose edge survives §5."
    )
    pos_c, neg_c = plots.DIVERGING[2][1], plots.DIVERGING[0][1]
    fig = go.Figure(
        go.Bar(
            y=[f"{g_} ({n} input{'' if n == 1 else 's'})" for g_, n in imp["inputs"].items()],
            x=imp["ic_drop"],
            orientation="h",
            name="IC drop",
            marker_color=[pos_c if v > 0 else neg_c for v in imp["ic_drop"]],
            error_x={"type": "data", "array": imp["spread"], "color": plots._MUTED},
        )
    )
    fig.add_vline(x=0, line={"color": plots._INK, "width": 1})
    plots._base(
        fig,
        title=f"Fig 4 · IC lost when one input group is shuffled (base IC "
        f"{imp_base:.4f}; blue = helps, red = hurts)",
        height=360,
    )
    fig.update_xaxes(title_text="IC drop (± its standard deviation over the shuffles)")
    fig.show()
else:
    imp, imp_base = None, None
    print("skipped (RUN_IMPORTANCE = False)")

## 5 · The coin-flip null (Table 5, Fig 5)

**Why a null.** An IC of 0.01 or a hit rate of 50.6% *sounds* small. Over 36,000 forecasts it could be real. It
could also be what a flexible model finds in any market, in three ways:
- by fitting noise;
- through the clock (volatility is higher at the open);
- through a leak somewhere in the pipeline.

Comparing with 0 or 50% cannot tell these apart.

**The coin-flip market** (`research.sign_surrogate`) is the real panel with every 1-minute bar flipped up or down
at random:
- One coin is thrown per minute for all five names, so their co-movement stays.
- It keeps every bar's size, range, volume and time of day, so every non-directional input looks as before. The
  option inputs are the real ones.
- Only direction is destroyed, so on it the right IC is 0 by construction.

The whole walk-forward (the same knobs, the same schedule) runs on `len(NULL_SEEDS)` coin-flip markets
(`research.null_study`, saved on disk like any walk-forward). Table 5b then puts the real result against them with
`research.null_z`:

z = min( (real − null mean) ÷ √(SE_real² + SE_null² ÷ k) , real t against 0 )

- Each SE is the run's own standard error, clustered by session (Table 2's t: SE = value ÷ t); SE_null is the mean
  over the k null runs. Each is read from 95 sessions, so it is well estimated. A spread of three null runs would
  not be: a standard deviation of three numbers has two degrees of freedom.
- **The verdict's gate:** z ≥ 2 on the IC and on the signed return.
- Two checks that the t of §2 is honest: the null runs' spread (`null sd (seeds)`) should be about one SE, and
  their own t values should scatter like ±1.
- The null's IC is 0 by construction (there is no direction to find), so the runs should average about 0; a mean
  far above 0 would reveal a leak (a §7 check). With a few seeds that mean is itself noisy: a null mean that lands
  below 0 by luck would lift the first term, which is why z is capped at the real result's t against exactly 0
  (Table 2's t). The gate has to pass both comparisons.

**Fig 5** draws every run with ± 2 SE. When the real market's bar overlaps the coin-flip runs, the forecaster cannot
be told apart from a model that had no direction to find.

In [ ]:
KEYS = ["ic", "t", "hit", "auc", "mse_skill", "sign_bp", "sign_t"]
SE_OF = {"ic": "t", "sign_bp": "sign_t"}  # metrics with a session-clustered t in research.summary
Z_SE, null = {}, None
if RUN_NULL:
    t0 = time.time()
    null = rs.null_study(panel, P, NULL_SEEDS, "tune", cost_bp=RT_BP).astype(float)
    table5 = pd.concat([null[KEYS], pd.DataFrame([{k: s2[k] for k in KEYS}], index=["real"])])
    table5.index = [f"coin-flip seed {i}" for i in null.index] + ["real market"]
    secs = time.time() - t0
    print(f"Table 5 · the real market against {len(null)} coin-flip markets ({secs:.0f} s)")
    display(table5.round(4))
    Z_SE = {m: rs.null_z(s2, null, m) for m in SE_OF}
    mu, sd = null.mean(), null.std(ddof=1)
    se_null = float((null["ic"] / null["t"]).abs().mean())  # SE of one null run's IC
    NULL_Z0 = float(mu["ic"] / (se_null / np.sqrt(len(null))))  # the null mean, in SEs from 0
    table5b = pd.DataFrame(
        [
            {
                "metric": m,
                "real": s2[m],
                "null mean": mu[m],
                "null sd (seeds)": sd[m],
                "SE real": abs(s2[m] / s2[SE_OF[m]]) if m in SE_OF else np.nan,
                "SE null (mean)": float((null[m] / null[SE_OF[m]]).abs().mean())
                if m in SE_OF
                else np.nan,
                "z (research.null_z)": Z_SE.get(m, np.nan),
            }
            for m in ("ic", "sign_bp", "hit", "auc")
        ]
    ).set_index("metric")
    print("Table 5b · the real market against the null (the verdict reads z)")
    display(table5b.round(4))
    print(
        f"z against the null: IC {Z_SE['ic']:+.2f}, signed bp {Z_SE['sign_bp']:+.2f}. "
        f"The null runs' own t of the IC: {', '.join(f'{v:+.2f}' for v in null['t'])} "
        f"(about ±1 when the session-clustered t is honest)."
    )
    print(
        f"The null's IC is 0 by construction; these {len(null)} runs average {mu['ic']:+.4f}, "
        f"{NULL_Z0:+.1f} standard errors from 0. z is capped at the real result's t against "
        f"exactly 0 (Table 2): IC {s2['t']:.2f}, signed bp {s2['sign_t']:.2f}."
    )
    fig = make_subplots(
        rows=1,
        cols=2,
        horizontal_spacing=0.1,
        subplot_titles=("rank IC ± 2 SE", "signed bp per forecast ± 2 SE"),
    )
    for j, m in enumerate(SE_OF, start=1):
        fig.add_trace(
            go.Scatter(
                x=[f"seed {s}" for s in null.index],
                y=null[m],
                mode="markers",
                name="coin-flip markets",
                showlegend=j == 1,
                marker={"color": plots._MUTED, "size": 9},
                error_y={
                    "type": "data",
                    "array": 2 * (null[m] / null[SE_OF[m]]).abs(),
                    "color": plots._MUTED,
                },
            ),
            row=1,
            col=j,
        )
        fig.add_trace(
            go.Scatter(
                x=["real"],
                y=[s2[m]],
                mode="markers",
                name="real market",
                showlegend=j == 1,
                marker={"color": plots.SERIES[0], "size": 13, "symbol": "diamond"},
                error_y={
                    "type": "data",
                    "array": [2 * table5b.loc[m, "SE real"]],
                    "color": plots.SERIES[0],
                },
            ),
            row=1,
            col=j,
        )
        fig.add_hline(
            y=mu[m], line={"color": plots._MUTED, "dash": "dash", "width": 1}, row=1, col=j
        )
        fig.add_hline(y=0, line={"color": plots._INK, "dash": "dot", "width": 1}, row=1, col=j)
    plots._base(
        fig,
        title="Fig 5 · the real market against the coin-flip markets (dashed: the null mean)",
        height=420,
    )
    fig.update_layout(margin={"t": 90})
    charts.below(fig, -0.12)
    fig.show()
else:
    print("skipped (RUN_NULL = False): §7 falls back on the session-clustered t alone")

## 6 · Other models on the same split (Table 6)

`COMPARE_MODELS` runs the same walk-forward (same split, schedule and inputs) for other models. Their §2 numbers
appear side by side with this notebook's model and §5's null. Each run is cached, so it is paid once.

**Run times**, measured on this machine with the defaults (5 names, 5-minute bars, a 40-session TRAIN slice;
`OMP_NUM_THREADS=2`, other notebooks running at the same time):

| model | one full fit | daily-retrained TUNE walk-forward (95 fits) |
|---|---|---|
| gbm, gbm_clf | about 5 s | about 8 min |
| tide (10 epochs) | about 5–8 s | about 8 min (measured: 95 fits, 8.4 min) |
| tft (10 epochs) | about 75 s | about 2 hours (estimated); about 15 min with `FINETUNE_EPOCHS = 1` |

To make the deep models cheaper, use any of these:
- `FINETUNE_EPOCHS = 1` or `2`: a full fit once, then a short fine-tune per retrain;
- `RETRAIN_MINUTES = 1950`: weekly retraining;
- `DEVICE = "mps"` on an Apple-silicon Mac;
- a shorter `TRAIN_DAYS`.

1-minute bars (`BAR_MINUTES = 1`) mean five times as many rows per fit. TiDE / TFT run without a VALIDATION
slice unless `VAL_DAYS > 0` (then early stopping decides the epochs, see §1).

A deep model's seed-to-seed spread is part of its null. Rerun it with another `seed` before believing a difference
between models.

In [ ]:
KEYS6 = ["ic", "t", "hit", "auc", "mse_skill", "sign_bp", "net_bp"]
rows = [
    {
        "model": MODEL,
        "fits": len(log),
        "fit_minutes": log["seconds"].sum() / 60,
        **{k: s2[k] for k in KEYS6},
    }
]
for m in COMPARE_MODELS:
    pm = {**P, "model": m}
    if m == MODEL or not SM["forecast_valid"](pm):
        print(
            f"{m}: skipped ({'already shown' if m == MODEL else 'not one of ' + str(SM['MODELS'])})"
        )
        continue
    t0 = time.time()
    fm = rs.cached_forecast(panel, pm)
    sm = rs.summary(rs.table(fm, "tune"), cost_bp=RT_BP)
    rows.append(
        {
            "model": m,
            "fits": len(fm["log"]),
            "fit_minutes": fm["log"]["seconds"].sum() / 60,
            **{k: sm.get(k, np.nan) for k in KEYS6},
        }
    )
    print(f"{m}: ready in {time.time() - t0:.0f} s")
if null is not None:
    rows.append({"model": "coin-flip null (mean)", **null[KEYS6].mean()})
table6 = pd.DataFrame(rows).set_index("model").astype(float)
print("Table 6 · models on the same TUNE walk-forward")
display(table6.round(4))
if not COMPARE_MODELS:
    print('COMPARE_MODELS is empty: set e.g. ["gbm_clf", "tide"] in §0 (mind the run times above)')

## 7 · Checks, the verdict and the scorecard

**The checks** confirm the walk-forward did what §1 says:
- every forecast was made at or after its model's retrain point, bar by bar, by a model fit only on bars before it;
- everything stayed inside TUNE;
- every planned fit ran, each on the `TRAIN_DAYS` sessions right before its retrain session, the last `VAL_DAYS` of
  them VALIDATION for tide/tft and none for gbm;
- the forecast scored is this panel's and these knobs' (the disk cache's key);
- §3's schedules forecast the same bars, and §4's static model reproduces §3's "once" forecasts;
- the coin-flip market is what §5 says it is, and its IC averages 0 within noise (no leak).

**The verdict** reads §2 and §5. A forecaster is **worth trading** only if both hold:
- it beats the coin-flip null: z ≥ 2 (`research.null_z`, Table 5b) on the IC and on the signed return;
- its signed return pays a round trip of costs: `net_bp > 0`, on all bars or on the strongest ones (Table 2c).

Only then does it go on to the strategy notebook (`darts_forecast.ipynb`: the trading rule, the Optuna study) and
to the TEST window. The verdict also repeats §3's answer on the retraining cadence. One `SCORECARD {json}` line
records the run.

In [ ]:
# bar level: the grid bar each forecast is made at vs the retrain point of the model that made it
bars, names_ = np.nonzero(np.isfinite(fc["edge"]))
retrain_at = np.full(len(fc["points"]), -1)
retrain_at[log["fit"].to_numpy(int)] = log["retrain_bucket"].to_numpy(int)
made_at = retrain_at[fc["fit"][bars, names_].astype(int)]
causal = bool((made_at >= 0).all() and (bars >= made_at).all())
a, b = rs.WINDOWS["tune"]
in_tune = bool(t["date"].min() >= a and t["date"].max() <= b)
window = fit_s - tr_a  # sessions from train_from up to the retrain session: TRAIN + VALIDATION
n_val = np.maximum(0, fit_s - tr_b - 1)  # sessions between the last fitted one and the retrain
want_val = VAL_DAYS if DEEP else 0
score_from = strat.optuna.get("score_from")
key = ["date", "minute", "name"]
same_bars = all(tb[key].equals(rt_tables[ref][key]) for tb in rt_tables.values())
# a saved forecast carries its knobs and code version; a fresh one only its panel id
mine = fc["panel_id"] == SM["fingerprint"](panel) and fc.get("params") in (None, rs.model_params(P))
checks = [
    (
        "every forecast is made at or after its model's retrain point "
        "(each model trains only on bars before that point)",
        causal,
        f"{len(bars):,} forecasts, {len(log)} fits, checked bar by bar",
    ),
    ("every scored forecast lies inside TUNE", in_tune, f"{t['date'].min()} → {t['date'].max()}"),
    (
        "every fit the schedule planned ran",
        len(log) == len(fc["points"]),
        f"{len(log)} of {len(fc['points'])}",
    ),
    (
        "every fit's window is the TRAIN_DAYS sessions right before its retrain session",
        bool((window == TRAIN_DAYS).all()),
        f"{window.min()}–{window.max()} sessions (TRAIN_DAYS {TRAIN_DAYS})",
    ),
    (
        "VALIDATION = the last VAL_DAYS of them for tide/tft, none for gbm",
        bool((n_val == want_val).all() and (log["val_days"] == want_val).all()),
        f"{n_val.min()}–{n_val.max()} VALIDATION sessions per fit (expected {want_val})",
    ),
    (
        "the first fit is config optuna.score_from (defaults only)",
        log["session"].iloc[0] == score_from or P0["train_days"] != TRAIN_DAYS,
        f"first fit {log['session'].iloc[0]}, score_from {score_from}",
    ),
    (
        "the forecast scored is this panel's and these knobs' (disk key: knobs, data, code)",
        bool(mine),
        f"panel {fc.get('panel_id')}, strategy.py {fc.get('code', rs.CODE_VERSION)}",
    ),
    (
        "every retraining schedule forecasts the same bars (§3 is paired)",
        same_bars,
        f"{len(rt_tables[ref]):,} scored forecasts in each of {len(rt_tables)} schedules",
    ),
]
if RUN_IMPORTANCE:
    checks.append(
        (
            "§4's static model reproduces §3's 'once' forecasts (same IC)",
            bool(abs(imp_base - rt.loc[0, "ic"]) < 1e-4),  # deep nets: float-level refit noise
            f"{imp_base:.6f} vs {rt.loc[0, 'ic']:.6f}",
        )
    )
if RUN_NULL:
    checks.append(
        (
            "the coin-flip runs' IC averages 0 within noise (a leak would push it up)",
            bool(abs(NULL_Z0) < 3),
            f"mean {null['ic'].mean():+.4f}, {NULL_Z0:+.2f} SE from 0 over {len(null)} runs",
        )
    )
    sur = rs.sign_surrogate(panel, NULL_SEEDS[0])
    r0 = np.diff(np.log(panel.close.to_numpy()), axis=0)
    r1 = np.diff(np.log(sur.close.to_numpy()), axis=0)
    ok = np.isfinite(r0) & np.isfinite(r1) & (r0 != 0)
    flipped = float((np.sign(r0[ok]) != np.sign(r1[ok])).mean())
    sur_ok = (
        np.allclose(sur.volume.to_numpy(), panel.volume.to_numpy(), equal_nan=True)
        and np.allclose(np.abs(r0[ok]), np.abs(r1[ok]))
        and 0.45 < flipped < 0.55
    )
    checks.append(
        (
            "the coin-flip market keeps every bar's volume and size and flips about half the moves",
            bool(sur_ok),
            f"{flipped:.1%} of the 1-minute moves flipped",
        )
    )
table7 = rs.checks_table(checks)
CHECKS_OK = bool(table7["ok"].all())
print("CHECKS")
with pd.option_context("display.max_colwidth", 200):
    display(table7)

best_tail = table2c.iloc[1:]["net_bp"].max()  # the strongest 50% ... 1%
if RUN_NULL:
    beats = Z_SE["ic"] >= 2 and Z_SE["sign_bp"] >= 2
    worse = Z_SE["ic"] <= -2
else:
    beats, worse = s2["t"] >= 2 and s2["sign_t"] >= 2, s2["t"] <= -2
pays = s2["net_bp"] > 0 or best_tail > 0
bar_ = "beats the coin-flip null" if RUN_NULL else "is significant (t ≥ 2; §5 not run)"
if beats and pays:
    verdict = "EDGE"
    words = f"{bar_} AND pays costs: take it to darts_forecast.ipynb, then TEST"
elif beats:
    verdict = "SIGNAL_NOT_TRADABLE"
    words = f"{bar_} but does not pay a round trip, not even on its strongest forecasts"
elif worse:
    verdict = "ANTI_SIGNAL"
    words = (
        "points the WRONG way, significantly; flipping it is a NEW hypothesis "
        "(choose it here, test it once on TEST)"
    )
else:
    verdict = "NO_EDGE"
    words = ("does not beat the coin-flip null" if RUN_NULL else "is not significant (t < 2)") + (
        ": no direction found in these inputs"
    )
src = "the coin-flip null" if RUN_NULL else "the session-clustered t (no null run)"
print(
    f"VERDICT · {MODEL}, {BAR_MINUTES}-minute bars, horizon {HORIZON}, retrain every "
    f"{RETRAIN_MINUTES} min: {verdict} ({words})\n"
    f"  IC {s2['ic']:.4f} (t {s2['t']:.2f}), hit {s2['hit']:.3f}, AUC {s2['auc']:.3f}, "
    f"signed {s2['sign_bp']:+.2f} bp per forecast, net {s2['net_bp']:+.2f} bp (round trip "
    f"{RT_BP:.1f}), best tail net {best_tail:+.2f} bp\n  judged against {src}"
    + (f": z IC {Z_SE['ic']:+.2f}, z signed bp {Z_SE['sign_bp']:+.2f}" if RUN_NULL else "")
    + f" · checks {'all OK' if CHECKS_OK else 'FAILED'}"
    + f"\n  retraining cadence (§3): {RETRAIN_WORDS}"
)
rs.scorecard(
    {
        "notebook": "darts_forecast_model",
        "model": MODEL,
        "bar_minutes": BAR_MINUTES,
        "horizon": HORIZON,
        "train_days": TRAIN_DAYS,
        "val_days": VAL_DAYS,
        "retrain_minutes": RETRAIN_MINUTES,
        "finetune_epochs": FINETUNE_EPOCHS,
        "options": OPTIONS,
        "window": list(rs.WINDOWS["tune"]),
        "fits": len(log),
        "fit_minutes": log["seconds"].sum() / 60,
        "forecasts": len(t),
        "sessions": int(t["date"].nunique()),
        **{k: s2[k] for k in KEYS6},
        "round_trip_bp": RT_BP,
        "best_tail_net_bp": best_tail,
        "null_ic": [null["ic"].mean(), null["ic"].std()] if null is not None else None,
        "z_se": Z_SE,
        "retrain_ic": {str(c): rt.loc[c, "ic"] for c in rt.index},
        "retrain_t_vs_once": {str(k): v for k, v in table3b["t"].items()},
        "retrain_differs": RETRAIN_DIFFERS,
        "importance": None if imp is None else imp["ic_drop"].to_dict(),
        "importance_base_ic": imp_base,
        "checks_ok": CHECKS_OK,
        "verdict": verdict,
    }
)